# SIRAP backend · RT-DETR Estados Unidos

Este notebook ejecuta detección de cajas con RT-DETRv2-S y OpenVINO Runtime nativo en Python. El OCR y las reglas de normalización específicas de Estados Unidos aún necesitan validación; por eso este flujo no declara lecturas de placas ni decisiones de acceso.

In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="usa"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Cargar detector del perfil USA
El IR local USA tiene prioridad. Si todavía no existe, usa el IR compartido de matrículas, que no ha sido validado para placas estadounidenses. En Colab, conviértelo antes con `python -m src.export_openvino --model-id merve/license-plates-rtdetrv2 --output-dir models/shared/detector/rtdetrv2_s/openvino`.

In [ ]:
import yaml
from src.model import PlateDetector
from src.data_processing import load_image
cfg=yaml.safe_load((REPO_ROOT/"config/countries/usa.yaml").read_text())["detector"]
shared_xml=REPO_ROOT/cfg["fallback_checkpoint"]/"openvino_model.xml"
if not shared_xml.is_file():
    from src.export_openvino import export_openvino_model
    export_openvino_model(MODEL_ID, shared_xml.parent)
detector=PlateDetector(checkpoint=cfg["checkpoint"],model_id=cfg["model_id"],
    imgsz=cfg["inference_size"],conf=cfg["confidence"],class_id=cfg["class_id"],
    device=cfg["device"],fallback_checkpoint=cfg.get("fallback_checkpoint"))
print(detector.name)
IMAGE_PATH=None
if IMAGE_PATH:
    image=load_image(Path(IMAGE_PATH).read_bytes())
    print([{"box":d.box,"confidence":d.confidence} for d in detector.detect(image)])


## Video de demostración
Guarda las cajas detectadas para inspección. Este archivo no contiene texto OCR ni una decisión de acceso.

In [ ]:
import cv2,pandas as pd
VIDEO_PATH="/content/SIRAP_workspace/video_camiones.mp4"
OUT="/content/detections_rtdetr_usa.csv"
cap=cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened(): raise FileNotFoundError(VIDEO_PATH)
fps=cap.get(cv2.CAP_PROP_FPS) or 25; step=max(1,int(fps*2.5)); rows=[]; index=0
while True:
    ok,frame=cap.read()
    if not ok: break
    if index%step==0:
        image=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        for det in detector.detect(image):
            rows.append({"frame_index":index,"x1":det.box[0],"y1":det.box[1],
                         "x2":det.box[2],"y2":det.box[3],"det_confidence":det.confidence})
    index+=1
cap.release(); pd.DataFrame(rows).to_csv(OUT,index=False)
print("Guardado",OUT,"detecciones",len(rows))
